# Análise Exploratória - Pinguins

Pipeline ETL implementado com DuckDB, Polars e Matplotlib para análise de dados de pinguins.

In [17]:
import duckdb
import polars as pl
from pathlib import Path
from typing import Union, Dict, List, Callable
import inspect
from dataclasses import dataclass

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

print("Bibliotecas importadas")

Bibliotecas importadas


## 2. Carregamento de Dados

Leitura do CSV usando DuckDB para parsing eficiente, conversão para Polars DataFrame e padronização de colunas.

In [18]:
def _standardize_columns(df: pl.DataFrame) -> pl.DataFrame:
    """Padronizar nomes de colunas em português para inglês."""
    cols = df.columns
    lower = [c.strip().lower() for c in cols]
    rename_map = {}

    depth_prefix = "profundidade do bico"
    idxs = [i for i, c in enumerate(lower) if c.startswith(depth_prefix)]
    if len(idxs) >= 2:
        for idx in idxs:
            series = df[cols[idx]].drop_nulls()
            try:
                median = float(series.median()) if len(series) > 0 else 0
            except Exception:
                median = 0
            if median and median > 100:
                rename_map[cols[idx]] = "flipper_length_mm"
            else:
                rename_map[cols[idx]] = "bill_depth_mm"

    for c in cols:
        lc = c.strip().lower()
        if lc == "espece":
            rename_map[c] = "species"
        elif lc == "ilha":
            rename_map[c] = "island"
        elif lc == "largura do bico":
            rename_map[c] = "bill_length_mm"
        elif lc == "massa corporal":
            rename_map[c] = "body_mass_g"
        elif lc == "sexo":
            rename_map[c] = "sex"

    df = df.rename(rename_map)
    return df


def load_penguins(path: Union[str, Path] = "dataset/penguins.csv") -> pl.DataFrame:
    """Carregar dataset de pinguins via DuckDB e retornar como Polars DataFrame."""
    path = str(path)
    safe_path = path.replace("'", "''")
    con = duckdb.connect()
    rel = con.execute(
        f"SELECT * FROM read_csv_auto('{safe_path}', nullstr=['NA', 'na', ''])"
    )
    df = pl.from_arrow(rel.arrow())
    df = _standardize_columns(df)

    casts = []
    for col in ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]:
        if col in df.columns:
            casts.append(pl.col(col).cast(pl.Float64))
    if casts:
        df = df.with_columns(casts)

    for c in ["species", "island", "sex"]:
        if c in df.columns:
            df = df.with_columns([pl.col(c).str.to_lowercase()])

    return df


df = load_penguins("dataset/penguins.csv")
print(f"Dataset carregado: {df.shape[0]} registros, {df.shape[1]} colunas")
print(f"Colunas: {df.columns}")

Dataset carregado: 344 registros, 7 colunas
Colunas: ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex']


## 3. Análises com Polars

In [19]:
def missing_summary(df: pl.DataFrame) -> pl.DataFrame:
    """Contabilizar valores faltantes por coluna."""
    missing_counts = [
        {"column": col, "missing_count": df[col].null_count()}
        for col in df.columns
    ]
    return pl.DataFrame(missing_counts).sort("missing_count", descending=True)


def rows_with_missing(df: pl.DataFrame) -> pl.DataFrame:
    """Retornar linhas com pelo menos um valor faltante."""
    exprs = [pl.col(c).is_null() for c in df.columns]
    return df.filter(pl.any_horizontal(exprs))


def island_counts(df: pl.DataFrame) -> pl.DataFrame:
    """Contar pinguins por ilha."""
    return df.group_by("island").agg(pl.len().alias("count")).sort("count", descending=True)


def species_counts(df: pl.DataFrame) -> pl.DataFrame:
    """Contar pinguins por espécie."""
    return df.group_by("species").agg(pl.len().alias("count")).sort("count", descending=True)


def species_stats(df: pl.DataFrame) -> pl.DataFrame:
    """Computar média de medidas por espécie."""
    return (
        df.group_by("species")
        .agg(
            pl.col("bill_length_mm").mean().alias("bill_length_mm_mean"),
            pl.col("bill_depth_mm").mean().alias("bill_depth_mm_mean"),
            pl.col("flipper_length_mm").mean().alias("flipper_length_mm_mean"),
            pl.col("body_mass_g").mean().alias("body_mass_g_mean"),
        )
        .sort("species")
    )


def sex_species_stats(df: pl.DataFrame) -> pl.DataFrame:
    """Computar média de medidas por espécie e sexo."""
    return (
        df.filter(pl.col("sex").is_not_null())
        .group_by("species", "sex")
        .agg(
            pl.len().alias("n"),
            pl.col("bill_length_mm").mean().alias("bill_length_mm_mean"),
            pl.col("bill_depth_mm").mean().alias("bill_depth_mm_mean"),
            pl.col("flipper_length_mm").mean().alias("flipper_length_mm_mean"),
            pl.col("body_mass_g").mean().alias("body_mass_g_mean"),
        )
        .sort("species", "sex")
    )


def compute_all_analyses(df: pl.DataFrame) -> Dict[str, pl.DataFrame]:
    """Executar todas as análises."""
    return {
        "missing_summary": missing_summary(df),
        "missing_rows": rows_with_missing(df),
        "island_counts": island_counts(df),
        "species_counts": species_counts(df),
        "species_stats": species_stats(df),
        "sex_species_stats": sex_species_stats(df),
    }


analyses = compute_all_analyses(df)
print("Análises calculadas")

Análises calculadas


## 4. Visualization Functions (Matplotlib + Seaborn)

In [20]:
# Configuração de tema e cores
sns.set_theme(style="whitegrid", context="talk")
PENGUIN_PALETTE = ["#1B4965", "#5FA8D3", "#62B6CB", "#CAE9FF", "#0B132B"]

def _save_fig(path: Path) -> str:
    """Salvar figura com qualidade alta."""
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    return str(path)


def _autopct_with_counts(values: List[int]) -> Callable:
    """Formatador para gráficos de pizza."""
    total = sum(values)
    def _formatter(pct: float) -> str:
        count = int(round(pct * total / 100.0))
        return f"{pct:.1f}% ({count})"
    return _formatter


def plot_missing_counts(missing_summary: pl.DataFrame, out_dir: Path) -> str:
    """Gráfico de barras: valores faltantes por coluna."""
    plt.figure(figsize=(10, 5))
    data = missing_summary.to_dict(as_series=False)
    n_cols = len(data["column"])
    palette = sns.color_palette("Blues", n_colors=n_cols)
    sns.barplot(x=data["column"], y=data["missing_count"], hue=data["column"],
                palette=palette, legend=False)
    plt.title("Número de dados faltantes por coluna")
    plt.xlabel("Coluna")
    plt.ylabel("Quantidade de faltantes")
    plt.xticks(rotation=20)
    return _save_fig(out_dir / "grafico_01_numero_dados_faltantes_por_coluna.png")


def plot_island_counts(island_counts: pl.DataFrame, out_dir: Path) -> str:
    """Gráfico de pizza: distribuição por ilha."""
    plt.figure(figsize=(8, 6))
    data = island_counts.to_dict(as_series=False)
    counts, islands = data["count"], data["island"]
    palette = sns.color_palette("Blues", n_colors=len(islands))
    wedges, _, _ = plt.pie(counts, labels=None, autopct=_autopct_with_counts(counts),
                            startangle=90, colors=palette, pctdistance=0.7,
                            wedgeprops={"edgecolor": "white", "linewidth": 1})
    plt.title("Distribuição de pinguins por ilha")
    legend_labels = [f"{name.title()} - {count}" for name, count in zip(islands, counts)]
    plt.legend(wedges, legend_labels, title="Ilhas", loc="center left",
               bbox_to_anchor=(1.02, 0.5), frameon=False)
    plt.axis("equal")
    return _save_fig(out_dir / "grafico_02_numero_de_pinguins_por_ilha.png")


def plot_species_counts(species_counts: pl.DataFrame, out_dir: Path) -> str:
    """Gráfico de pizza: distribuição por espécie."""
    plt.figure(figsize=(8, 6))
    data = species_counts.to_dict(as_series=False)
    counts, species = data["count"], data["species"]
    palette = sns.color_palette("Blues", n_colors=len(species))
    wedges, _, _ = plt.pie(counts, labels=None, autopct=_autopct_with_counts(counts),
                            startangle=90, colors=palette, pctdistance=0.7,
                            wedgeprops={"edgecolor": "white", "linewidth": 1})
    plt.title("Distribuição de pinguins por espécie")
    legend_labels = [f"{name.capitalize()} - {count}" for name, count in zip(species, counts)]
    plt.legend(wedges, legend_labels, title="Espécies", loc="center left",
               bbox_to_anchor=(1.02, 0.5), frameon=False)
    plt.axis("equal")
    return _save_fig(out_dir / "grafico_03_numero_de_pinguins_por_especie.png")


def plot_scatter_measures_by_species(df: pl.DataFrame, out_dir: Path) -> str:
    """Scatter plot: medidas do bico por espécie."""
    plt.figure(figsize=(9, 6))
    data = df.to_dict(as_series=False)
    sns.scatterplot(x=data["bill_length_mm"], y=data["bill_depth_mm"],
                    hue=data["species"], palette=PENGUIN_PALETTE[:3], alpha=0.85)
    plt.title("Comprimento vs profundidade do bico por espécie")
    plt.xlabel("Comprimento do bico (mm)")
    plt.ylabel("Profundidade do bico (mm)")
    return _save_fig(out_dir / "grafico_04_relacao_medidas_por_especie.png")


def plot_pairplot_species(df: pl.DataFrame, out_dir: Path) -> str:
    """Pairplot: relações entre muitas variáveis por espécie."""
    cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
    clean_df = df.select(["species"] + cols).drop_nulls()
    data = clean_df.to_dict(as_series=False)
    
    n_cols = len(cols)
    fig, axes = plt.subplots(n_cols, n_cols, figsize=(12, 12))
    species_list = list(set(data["species"]))
    species_colors = {sp: PENGUIN_PALETTE[i % len(PENGUIN_PALETTE)] for i, sp in enumerate(species_list)}
    
    for i, col_y in enumerate(cols):
        for j, col_x in enumerate(cols):
            ax = axes[i, j]
            if i == j:
                for species in species_list:
                    mask = [s == species for s in data["species"]]
                    values = [data[col_x][k] for k in range(len(data["species"])) if mask[k]]
                    ax.hist(values, alpha=0.5, label=species, color=species_colors[species], bins=10)
                if i == 0:
                    ax.legend(fontsize=8)
            else:
                for species in species_list:
                    mask = [s == species for s in data["species"]]
                    x_vals = [data[col_x][k] for k in range(len(data["species"])) if mask[k]]
                    y_vals = [data[col_y][k] for k in range(len(data["species"])) if mask[k]]
                    ax.scatter(x_vals, y_vals, alpha=0.6, label=species, color=species_colors[species], s=30)
            
            if i == n_cols - 1:
                ax.set_xlabel(col_x, fontsize=9)
            else:
                ax.set_xticklabels([])
            if j == 0:
                ax.set_ylabel(col_y, fontsize=9)
            else:
                ax.set_yticklabels([])
    
    fig.suptitle("Relações entre medidas por espécie", fontsize=14)
    path = out_dir / "grafico_05_pairplot_medidas_por_especie.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return str(path)


def plot_mass_by_sex_species(df: pl.DataFrame, out_dir: Path) -> str:
    """Boxplot: massa corporal por sexo em cada espécie."""
    clean_df = df.filter(pl.col("sex").is_not_null()).select(["species", "sex", "body_mass_g"])
    species_list = ["adelie", "gentoo", "chinstrap"]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    palette = sns.color_palette("Blues", n_colors=2)
    
    for idx, species in enumerate(species_list):
        ax = axes[idx]
        species_data = clean_df.filter(pl.col("species") == species)
        if species_data.height == 0:
            continue
        
        data_dict = species_data.to_dict(as_series=False)
        males = [data_dict["body_mass_g"][i] for i in range(len(data_dict["sex"])) if data_dict["sex"][i] == "male"]
        females = [data_dict["body_mass_g"][i] for i in range(len(data_dict["sex"])) if data_dict["sex"][i] == "female"]
        
        box_data = []
        labels = []
        if males:
            box_data.append(males)
            labels.append("male")
        if females:
            box_data.append(females)
            labels.append("female")
        
        if box_data:
            bp = ax.boxplot(box_data, labels=labels, patch_artist=True)
            for patch, color in zip(bp["boxes"], palette[:len(box_data)]):
                patch.set_facecolor(color)
        
        ax.set_title(f"{species.capitalize()}", fontweight="bold")
        ax.set_ylabel("Massa corporal (g)")
        ax.set_xlabel("Sexo")
        ax.grid(True, alpha=0.3, axis="y")
    
    fig.suptitle("Massa corporal por sexo em cada espécie", fontsize=14)
    path = out_dir / "grafico_06_massa_por_sexo_em_cada_especie.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return str(path)


def plot_pairplot_by_species_sex(df: pl.DataFrame, out_dir: Path) -> List[str]:
    """Pairplot: medidas por sexo em cada espécie."""
    files = []
    cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
    species_order = ["adelie", "gentoo", "chinstrap"]
    palette = sns.color_palette("Blues", n_colors=2)
    
    for file_idx, species in enumerate(species_order, start=7):
        species_df = df.filter((pl.col("species") == species) & (pl.col("sex").is_not_null())).select(["sex"] + cols)
        if species_df.height == 0:
            continue
        
        data_dict = species_df.to_dict(as_series=False)
        n_cols = len(cols)
        fig, axes = plt.subplots(n_cols, n_cols, figsize=(11, 11))
        sex_colors = {"male": palette[0], "female": palette[1]}
        
        for i, col_y in enumerate(cols):
            for j, col_x in enumerate(cols):
                ax = axes[i, j]
                if i == j:
                    for sex_val in ["male", "female"]:
                        mask = [s == sex_val for s in data_dict["sex"]]
                        values = [data_dict[col_x][k] for k in range(len(data_dict["sex"])) if mask[k]]
                        ax.hist(values, alpha=0.5, label=sex_val, color=sex_colors[sex_val], bins=8)
                    if i == 0:
                        ax.legend(fontsize=8)
                else:
                    for sex_val in ["male", "female"]:
                        mask = [s == sex_val for s in data_dict["sex"]]
                        x_vals = [data_dict[col_x][k] for k in range(len(data_dict["sex"])) if mask[k]]
                        y_vals = [data_dict[col_y][k] for k in range(len(data_dict["sex"])) if mask[k]]
                        ax.scatter(x_vals, y_vals, alpha=0.6, label=sex_val, color=sex_colors[sex_val], s=30)
                
                if i == n_cols - 1:
                    ax.set_xlabel(col_x, fontsize=9)
                else:
                    ax.set_xticklabels([])
                if j == 0:
                    ax.set_ylabel(col_y, fontsize=9)
                else:
                    ax.set_yticklabels([])
        
        title = f"Relação entre medidas e sexo - {species.capitalize()}"
        fig.suptitle(title, fontsize=12)
        path = out_dir / f"grafico_{file_idx:02d}_pairplot_{species}_sexo.png"
        fig.savefig(path, dpi=300, bbox_inches="tight")
        plt.close(fig)
        files.append(str(path))
    
    return files

print("Funções de visualização definidas")

Funções de visualização definidas


## 5. Execução Completa do Pipeline ETL

In [21]:
out_dir = Path("outputs/graficos")
out_dir.mkdir(parents=True, exist_ok=True)

print("Gerando gráficos...")
graph_paths = []

file1 = plot_missing_counts(analyses["missing_summary"], out_dir)
graph_paths.append(file1)
print(f"  Gráfico 1: {Path(file1).name}")

file2 = plot_island_counts(analyses["island_counts"], out_dir)
graph_paths.append(file2)
print(f"  Gráfico 2: {Path(file2).name}")

file3 = plot_species_counts(analyses["species_counts"], out_dir)
graph_paths.append(file3)
print(f"  Gráfico 3: {Path(file3).name}")

clean_for_scatter = df.drop_nulls(["bill_length_mm", "bill_depth_mm", "species"])
file4 = plot_scatter_measures_by_species(clean_for_scatter, out_dir)
graph_paths.append(file4)
print(f"  Gráfico 4: {Path(file4).name}")

file5 = plot_pairplot_species(df, out_dir)
graph_paths.append(file5)
print(f"  Gráfico 5: {Path(file5).name}")

file6 = plot_mass_by_sex_species(df, out_dir)
graph_paths.append(file6)
print(f"  Gráfico 6: {Path(file6).name}")

files_sex = plot_pairplot_by_species_sex(df, out_dir)
graph_paths.extend(files_sex)
print(f"  Gráficos 7-9: {len(files_sex)} pairplots")

parquet_path = Path("outputs/penguins_limpo.parquet")
parquet_path.parent.mkdir(parents=True, exist_ok=True)
df.write_parquet(parquet_path)
print(f"\nPipeline concluído. {len(graph_paths)} gráficos gerados.")

Gerando gráficos...
  Gráfico 1: grafico_01_numero_dados_faltantes_por_coluna.png
  Gráfico 2: grafico_02_numero_de_pinguins_por_ilha.png
  Gráfico 3: grafico_03_numero_de_pinguins_por_especie.png
  Gráfico 4: grafico_04_relacao_medidas_por_especie.png
  Gráfico 5: grafico_05_pairplot_medidas_por_especie.png


C:\Users\lucen\AppData\Local\Temp\ipykernel_8588\2540248347.py:154: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(box_data, labels=labels, patch_artist=True)
C:\Users\lucen\AppData\Local\Temp\ipykernel_8588\2540248347.py:154: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(box_data, labels=labels, patch_artist=True)
C:\Users\lucen\AppData\Local\Temp\ipykernel_8588\2540248347.py:154: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(box_data, labels=labels, patch_artist=True)


  Gráfico 6: grafico_06_massa_por_sexo_em_cada_especie.png
  Gráficos 7-9: 3 pairplots

Pipeline concluído. 9 gráficos gerados.


## 6. Resoluções das Perguntas

### PERGUNTA 1: Quais pinguins não têm anotações?

In [22]:
print("=== PERGUNTA 1: Dados faltantes ===")
print(analyses["missing_summary"])
print(f"\nTotal de registros com dados faltantes: {len(analyses['missing_rows'])}")
print(f"Completude do dataset: {(1 - len(analyses['missing_rows'])/len(df)) * 100:.2f}%")

=== PERGUNTA 1: Dados faltantes ===
shape: (7, 2)
┌───────────────────┬───────────────┐
│ column            ┆ missing_count │
│ ---               ┆ ---           │
│ str               ┆ i64           │
╞═══════════════════╪═══════════════╡
│ sex               ┆ 11            │
│ bill_length_mm    ┆ 2             │
│ bill_depth_mm     ┆ 2             │
│ flipper_length_mm ┆ 2             │
│ body_mass_g       ┆ 2             │
│ species           ┆ 0             │
│ island            ┆ 0             │
└───────────────────┴───────────────┘

Total de registros com dados faltantes: 11
Completude do dataset: 96.80%


### PERGUNTA 2: Quais ilhas a maioria dos pinguins está vindo?

In [23]:
print("\n=== PERGUNTA 2: Distribuição por ilha ===")
print(analyses["island_counts"])
island_data = analyses["island_counts"]
total = island_data["count"].sum()
print("\nProporções:")
for row in island_data.to_dicts():
    pct = (row["count"] / total) * 100
    print(f"  {row['island'].title()}: {row['count']} ({pct:.2f}%)")


=== PERGUNTA 2: Distribuição por ilha ===
shape: (3, 2)
┌───────────┬───────┐
│ island    ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ biscoe    ┆ 168   │
│ dream     ┆ 124   │
│ torgersen ┆ 52    │
└───────────┴───────┘

Proporções:
  Biscoe: 168 (48.84%)
  Dream: 124 (36.05%)
  Torgersen: 52 (15.12%)


### PERGUNTA 3: Quais espécies a ONG mais possui?

In [24]:
print("\n=== PERGUNTA 3: Distribuição por espécie ===")
print(analyses["species_counts"])
species_data = analyses["species_counts"]
total = species_data["count"].sum()
print("\nProporções:")
for row in species_data.to_dicts():
    pct = (row["count"] / total) * 100
    print(f"  {row['species'].capitalize()}: {row['count']} ({pct:.2f}%)")


=== PERGUNTA 3: Distribuição por espécie ===
shape: (3, 2)
┌───────────┬───────┐
│ species   ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ adelie    ┆ 152   │
│ gentoo    ┆ 124   │
│ chinstrap ┆ 68    │
└───────────┴───────┘

Proporções:
  Adelie: 152 (44.19%)
  Gentoo: 124 (36.05%)
  Chinstrap: 68 (19.77%)


### PERGUNTA 4: Existe relação entre as medidas do pinguim e sua espécie?

In [29]:
print("\n=== PERGUNTA 4: Relação entre medidas e espécie ===")
numeric_cols = [c for c in analyses["species_stats"].columns if c != "species"]
print(analyses["species_stats"].with_columns([pl.col(c).round(2) for c in numeric_cols]))


=== PERGUNTA 4: Relação entre medidas e espécie ===
shape: (3, 5)
┌───────────┬─────────────────────┬────────────────────┬────────────────────────┬──────────────────┐
│ species   ┆ bill_length_mm_mean ┆ bill_depth_mm_mean ┆ flipper_length_mm_mean ┆ body_mass_g_mean │
│ ---       ┆ ---                 ┆ ---                ┆ ---                    ┆ ---              │
│ str       ┆ f64                 ┆ f64                ┆ f64                    ┆ f64              │
╞═══════════╪═════════════════════╪════════════════════╪════════════════════════╪══════════════════╡
│ adelie    ┆ 38.79               ┆ 18.35              ┆ 189.95                 ┆ 3700.66          │
│ chinstrap ┆ 48.83               ┆ 18.42              ┆ 195.82                 ┆ 3733.09          │
│ gentoo    ┆ 47.5                ┆ 14.98              ┆ 217.19                 ┆ 5076.02          │
└───────────┴─────────────────────┴────────────────────┴────────────────────────┴──────────────────┘


### PERGUNTA 5: Existe relação entre as medidas e sexo para cada espécie?

In [30]:
print("\n=== PERGUNTA 5: Relação entre medidas, sexo e espécie ===")
numeric_cols = [c for c in analyses["sex_species_stats"].columns if c not in ["species", "sex"]]
print(analyses["sex_species_stats"].with_columns([pl.col(c).round(2) for c in numeric_cols]))


=== PERGUNTA 5: Relação entre medidas, sexo e espécie ===
shape: (6, 7)
┌───────────┬────────┬─────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┐
│ species   ┆ sex    ┆ n   ┆ bill_length_mm_ ┆ bill_depth_mm_m ┆ flipper_length_ ┆ body_mass_g_mea │
│ ---       ┆ ---    ┆ --- ┆ mean            ┆ ean             ┆ mm_mean         ┆ n               │
│ str       ┆ str    ┆ u32 ┆ ---             ┆ ---             ┆ ---             ┆ ---             │
│           ┆        ┆     ┆ f64             ┆ f64             ┆ f64             ┆ f64             │
╞═══════════╪════════╪═════╪═════════════════╪═════════════════╪═════════════════╪═════════════════╡
│ adelie    ┆ female ┆ 73  ┆ 37.26           ┆ 17.62           ┆ 187.79          ┆ 3368.84         │
│ adelie    ┆ male   ┆ 73  ┆ 40.39           ┆ 19.07           ┆ 192.41          ┆ 4043.49         │
│ chinstrap ┆ female ┆ 34  ┆ 46.57           ┆ 17.59           ┆ 191.74          ┆ 3527.21         │
│ chinstrap ┆ male

## 7. Visualização dos Gráficos Gerados

## 8. Resumo dos Resultados

In [32]:
for i, graph_path in enumerate(graph_paths, start=1):
    p = Path(graph_path)
    print(f"Gráfico {i}: {p.name}")
    img = mpimg.imread(p)
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Gráfico {i}: {p.name}", fontsize=11)
    plt.tight_layout()
    plt.show()

Gráfico 1: grafico_01_numero_dados_faltantes_por_coluna.png


C:\Users\lucen\AppData\Local\Temp\ipykernel_8588\3529661278.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Gráfico 2: grafico_02_numero_de_pinguins_por_ilha.png
Gráfico 3: grafico_03_numero_de_pinguins_por_especie.png
Gráfico 4: grafico_04_relacao_medidas_por_especie.png
Gráfico 5: grafico_05_pairplot_medidas_por_especie.png
Gráfico 6: grafico_06_massa_por_sexo_em_cada_especie.png
Gráfico 7: grafico_07_pairplot_adelie_sexo.png
Gráfico 8: grafico_08_pairplot_gentoo_sexo.png
Gráfico 9: grafico_09_pairplot_chinstrap_sexo.png


In [ ]:
print("\n" + "="*70)
print("RESUMO DOS RESULTADOS")
print("="*70 + "\n")

print("PERGUNTA 1: Dados Faltantes")
print("-" * 70)
missing_summary = analyses["missing_summary"]
total_missing = len(analyses["missing_rows"])
completeness = (1 - total_missing / len(df)) * 100
print(f"Total de valores faltantes: {total_missing}/344 ({100-completeness:.2f}%)")
print(f"Completude do dataset: {completeness:.2f}%")
print("\nMaiores lacunas:")
print(missing_summary.head(3))




RESUMO DOS RESULTADOS

PERGUNTA 1: Dados Faltantes
----------------------------------------------------------------------
Total de valores faltantes: 11/344 (3.20%)
Completude do dataset: 96.80%

Maiores lacunas:
shape: (3, 2)
┌────────────────┬───────────────┐
│ column         ┆ missing_count │
│ ---            ┆ ---           │
│ str            ┆ i64           │
╞════════════════╪═══════════════╡
│ sex            ┆ 11            │
│ bill_length_mm ┆ 2             │
│ bill_depth_mm  ┆ 2             │
└────────────────┴───────────────┘


PERGUNTA 2: Distribuição por Ilha
----------------------------------------------------------------------
shape: (3, 2)
┌───────────┬───────┐
│ island    ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ biscoe    ┆ 168   │
│ dream     ┆ 124   │
│ torgersen ┆ 52    │
└───────────┴───────┘


PERGUNTA 3: Distribuição por Espécie
----------------------------------------------------------------------
shape: (3, 2)
┌───────────┬─